# 🔍 Company Deep Dive

Interactive company analysis — live data from OpenBB.

**Usage:** Enter a ticker, run all cells. Get: profile, financials, charts, news.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from src.data_engine import fetch_all_for_ticker, get_price_history

plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100

print('✅ Data engine loaded.')

## 1. Select Company

In [ ]:
TICKER = 'NVDA'  # ← Change this to analyze any company

print(f'Fetching all data for {TICKER}...')
data = fetch_all_for_ticker(TICKER, days_of_news=30)

profile = data.get('profile')
metrics = data.get('metrics')
income = data.get('income')
balance = data.get('balance')
cashflow = data.get('cashflow')
estimates = data.get('estimates')
news = data.get('news')
filings = data.get('filings')
price_history = data.get('price_history')

print(f'✅ Data fetched. Ticker: {TICKER}')

## 2. Company Profile

In [ ]:
if profile is not None and not profile.empty:
    p = profile.iloc[0]
    print(f"{'='*50}")
    print(f"  {p.get('name', TICKER)} ({TICKER})")
    print(f"{'='*50}")
    for key in ['sector', 'industry', 'market_cap', 'full_time_employees', 'country', 'website', 'summary']:
        if key in profile.columns:
            val = p[key]
            if key == 'market_cap' and val:
                print(f"  Market Cap: ${val/1e12:.2f}T" if val > 1e12 else f"  Market Cap: ${val/1e9:.0f}B")
            elif key == 'summary' and val:
                print(f"\n  {str(val)[:500]}...")
            else:
                print(f"  {key}: {val}")
else:
    print('⚠️  No profile data.')

## 3. Key Valuation Metrics

In [ ]:
if metrics is not None and not metrics.empty:
    m = metrics.iloc[0]
    key_cols = ['pe_ratio', 'pb_ratio', 'ps_ratio', 'ev_to_ebitda',
                'roe', 'roa', 'gross_margin', 'operating_margin', 'net_margin',
                'revenue_growth', 'debt_to_equity', 'current_ratio']
    available = [(c, m[c]) for c in key_cols if c in metrics.columns and pd.notna(m[c])]
    for label, val in available:
        label_fmt = label.replace('_', ' ').title()
        if 'margin' in label or 'roe' in label or 'roa' in label or 'growth' in label:
            print(f"  {label_fmt:<25} {val:>8.1%}")
        elif 'debt' in label or 'current' in label:
            print(f"  {label_fmt:<25} {val:>8.2f}")
        else:
            print(f"  {label_fmt:<25} {val:>8.1f}x")
else:
    print('⚠️  No metrics available.')

## 4. Revenue & Earnings Trend

In [ ]:
if income is not None and not income.empty:
    inc = income.copy()
    # Sort by period ending ascending
    if 'period_ending' in inc.columns:
        inc['period_ending'] = pd.to_datetime(inc['period_ending'])
        inc = inc.sort_values('period_ending')
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Revenue
    ax = axes[0]
    if 'total_revenue' in inc.columns and 'period_ending' in inc.columns:
        rev = inc[['period_ending', 'total_revenue']].dropna()
        ax.bar(range(len(rev)), rev['total_revenue'] / 1e9, color='steelblue', alpha=0.8)
        ax.set_xticks(range(len(rev)))
        ax.set_xticklabels([d.strftime('%Y') for d in rev['period_ending']], rotation=45)
        ax.set_title(f'{TICKER} — Revenue')
        ax.set_ylabel('Revenue ($B)')
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.0f}B'))
    
    # Net Income + Operating Income
    ax = axes[1]
    if 'period_ending' in inc.columns:
        plot_data = inc[['period_ending']].copy()
        has_ni = 'net_income' in inc.columns
        has_oi = 'operating_income' in inc.columns
        if has_ni:
            plot_data['net_income'] = inc['net_income'] / 1e9
            ax.plot(range(len(plot_data)), plot_data['net_income'], 'o-', label='Net Income', color='green', linewidth=2)
        if has_oi:
            plot_data['operating_income'] = inc['operating_income'] / 1e9
            ax.plot(range(len(plot_data)), plot_data['operating_income'], 's--', label='Operating Income', color='orange', linewidth=2)
        if has_ni or has_oi:
            ax.set_xticks(range(len(plot_data)))
            ax.set_xticklabels([d.strftime('%Y') for d in plot_data['period_ending']], rotation=45)
            ax.set_title(f'{TICKER} — Earnings')
            ax.set_ylabel('$ Billions')
            ax.legend()
            ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.0f}B'))
    
    plt.tight_layout()
    plt.show()
else:
    print('⚠️  No income statement data.')

## 5. Cash Flow & FCF

In [ ]:
if cashflow is not None and not cashflow.empty:
    cf = cashflow.copy()
    if 'period_ending' in cf.columns:
        cf['period_ending'] = pd.to_datetime(cf['period_ending'])
        cf = cf.sort_values('period_ending')
    
    # Show available columns for reference
    available = [c for c in ['operating_cash_flow', 'capital_expenditure', 'free_cash_flow'] if c in cf.columns]
    print(f"Available CF columns: {available}")
    display(cf[['period_ending'] + available].head(5) if 'period_ending' in cf.columns else cf[available].head(5))
else:
    print('⚠️  No cash flow data.')

## 6. Price Chart (K-Line)

In [ ]:
if price_history is not None and not price_history.empty:
    ph = price_history.copy()
    if 'date' in ph.columns:
        ph['date'] = pd.to_datetime(ph['date'])
        ph = ph.set_index('date')
    
    fig, ax = plt.subplots(figsize=(14, 6))
    if 'close' in ph.columns:
        ax.plot(ph.index, ph['close'], linewidth=1.5, color='navy', label='Close')
        ax.fill_between(ph.index, ph['close'].min(), ph['close'], alpha=0.3, color='navy')
        ax.set_title(f'{TICKER} — Price Chart (1 Year)')
        ax.set_ylabel('Price ($)')
        ax.legend()
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print('⚠️  No price history.')

## 7. Recent News

In [ ]:
if news is not None and not news.empty:
    for _, article in news.head(10).iterrows():
        title = article.get('title', 'N/A')
        source = article.get('source', '')
        date = article.get('date', '')
        print(f"📰 {title}")
        print(f"   {source} · {date}")
        print()
else:
    print('⚠️  No recent news.')

## 8. SEC Filings

In [ ]:
if filings is not None and not filings.empty:
    display_cols = ['filing_date', 'form_type', 'description']
    available = [c for c in display_cols if c in filings.columns]
    display(filings[available].head(10))
else:
    print('⚠️  No SEC filing data.')

---
*Generated by AI Investment System — {{ datetime.now().strftime('%Y-%m-%d') }}*